# DE Data Pipeline
This notebook loads raw CSV data, applies schema typing, performs business transformations, and answers analytical questions.

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, BooleanType, DoubleType, TimestampType, DateType

base_path = "/Volumes/enterprisedata_test/default/sandbox"

# Explicit schemas avoid the extra data pass that inferSchema requires
products_schema = StructType([
    StructField("ProductID", IntegerType()),
    StructField("ProductDesc", StringType()),
    StructField("ProductNumber", StringType()),
    StructField("MakeFlag", BooleanType()),
    StructField("Color", StringType()),
    StructField("SafetyStockLevel", IntegerType()),
    StructField("ReorderPoint", IntegerType()),
    StructField("StandardCost", DoubleType()),
    StructField("ListPrice", DoubleType()),
    StructField("Size", StringType()),
    StructField("SizeUnitMeasureCode", StringType()),
    StructField("Weight", DoubleType()),
    StructField("WeightUnitMeasureCode", StringType()),
    StructField("ProductCategoryName", StringType()),
    StructField("ProductSubCategoryName", StringType()),
])

detail_schema = StructType([
    StructField("SalesOrderID", IntegerType()),
    StructField("SalesOrderDetailID", IntegerType()),
    StructField("OrderQty", IntegerType()),
    StructField("ProductID", IntegerType()),
    StructField("UnitPrice", DoubleType()),
    StructField("UnitPriceDiscount", DoubleType()),
])

header_schema = StructType([
    StructField("SalesOrderID", IntegerType()),
    StructField("OrderDate", TimestampType()),
    StructField("ShipDate", DateType()),
    StructField("OnlineOrderFlag", BooleanType()),
    StructField("AccountNumber", StringType()),
    StructField("CustomerID", IntegerType()),
    StructField("SalesPersonID", IntegerType()),
    StructField("Freight", DoubleType()),
])

# Load with explicit schemas — single pass, no inferSchema overhead
raw_products = spark.read.option("header", True).schema(products_schema).csv(f"{base_path}/products.csv")
raw_products.createOrReplaceTempView("raw_products")

raw_sales_order_detail = spark.read.option("header", True).schema(detail_schema).csv(f"{base_path}/sales_order_detail.csv")
raw_sales_order_detail.createOrReplaceTempView("raw_sales_order_detail")

raw_sales_order_header = spark.read.option("header", True).schema(header_schema).csv(f"{base_path}/sales_order_header.csv")
raw_sales_order_header.createOrReplaceTempView("raw_sales_order_header")

# Display samples avoid unnecessary full scans
print("=== raw_products ===")
display(raw_products.limit(5))
print("=== raw_sales_order_detail ===")
display(raw_sales_order_detail.limit(5))
print("=== raw_sales_order_header ===")
display(raw_sales_order_header.limit(5))

=== raw_products ===


ProductID,ProductDesc,ProductNumber,MakeFlag,Color,SafetyStockLevel,ReorderPoint,StandardCost,ListPrice,Size,SizeUnitMeasureCode,Weight,WeightUnitMeasureCode,ProductCategoryName,ProductSubCategoryName
680,"HL Road Frame - Black, 58",FR-R92B-58,true,Black,500,375,1059.31,1431.5,58,CM,2.24,LB,null,Road Frames
706,"HL Road Frame - Red, 58",FR-R92R-58,true,Red,500,375,1059.31,1431.5,58,CM,2.24,LB,null,Road Frames
707,"Sport-100 Helmet, Red",HL-U509-R,false,Red,4,3,13.0863,34.99,null,null,null,null,null,Helmets
708,"Sport-100 Helmet, Black",HL-U509,false,Black,4,3,13.0863,34.99,null,null,null,null,null,Helmets
709,"Mountain Bike Socks, M",SO-B909-M,false,White,4,3,3.3963,9.5,M,null,null,null,null,Socks


=== raw_sales_order_detail ===


SalesOrderID,SalesOrderDetailID,OrderQty,ProductID,UnitPrice,UnitPriceDiscount
43659,1,1,776,2024.994,0.0
43659,2,3,777,2024.994,0.0
43659,3,1,778,2024.994,0.0
43659,4,1,771,2039.994,0.0
43659,5,1,772,2039.994,0.0


=== raw_sales_order_header ===


SalesOrderID,OrderDate,ShipDate,OnlineOrderFlag,AccountNumber,CustomerID,SalesPersonID,Freight
43828,2021-06-01T00:00:00.000Z,2021-07-05,true,10-4030-027605,27605,null,89.4568
43829,2021-06-01T00:00:00.000Z,2021-07-05,true,10-4030-027611,27611,null,89.4568
43830,2021-06-01T00:00:00.000Z,2021-07-05,true,10-4030-016347,16347,null,89.4568
43831,2021-06-01T00:00:00.000Z,2021-07-05,true,10-4030-011028,11028,null,84.3748
43832,2021-06-01T00:00:00.000Z,2021-07-05,true,10-4030-013584,13584,null,89.4568


## Data Review & Storage
Cast columns to appropriate data types, identify primary and foreign keys, and save as Delta tables with the store_ prefix.

In [0]:
# Store products with proper data types
# PK: ProductID
spark.sql("""
CREATE OR REPLACE TABLE enterprisedata_test.default.store_products AS
SELECT
  CAST(ProductID AS INT) AS ProductID,     -- PK
  CAST(ProductDesc AS STRING) AS ProductDesc,
  CAST(ProductNumber AS STRING) AS ProductNumber,
  CAST(MakeFlag AS BOOLEAN) AS MakeFlag,
  CAST(Color AS STRING) AS Color,
  CAST(SafetyStockLevel AS SHORT) AS SafetyStockLevel,
  CAST(ReorderPoint AS SHORT) AS ReorderPoint,
  CAST(StandardCost AS DECIMAL(10,4)) AS StandardCost,
  CAST(ListPrice AS DECIMAL(10,4)) AS ListPrice,
  CAST(Size AS STRING) AS Size,
  CAST(SizeUnitMeasureCode AS STRING) AS SizeUnitMeasureCode,
  CAST(Weight AS DECIMAL(8,2)) AS Weight,
  CAST(WeightUnitMeasureCode AS STRING) AS WeightUnitMeasureCode,
  CAST(ProductCategoryName AS STRING) AS ProductCategoryName,
  CAST(ProductSubCategoryName AS STRING) AS ProductSubCategoryName
FROM raw_products
""")

display(spark.table("enterprisedata_test.default.store_products").limit(10))

ProductID,ProductDesc,ProductNumber,MakeFlag,Color,SafetyStockLevel,ReorderPoint,StandardCost,ListPrice,Size,SizeUnitMeasureCode,Weight,WeightUnitMeasureCode,ProductCategoryName,ProductSubCategoryName
680,"HL Road Frame - Black, 58",FR-R92B-58,true,Black,500,375,1059.3100,1431.5000,58,CM,2.24,LB,null,Road Frames
706,"HL Road Frame - Red, 58",FR-R92R-58,true,Red,500,375,1059.3100,1431.5000,58,CM,2.24,LB,null,Road Frames
707,"Sport-100 Helmet, Red",HL-U509-R,false,Red,4,3,13.0863,34.9900,null,null,null,null,null,Helmets
708,"Sport-100 Helmet, Black",HL-U509,false,Black,4,3,13.0863,34.9900,null,null,null,null,null,Helmets
709,"Mountain Bike Socks, M",SO-B909-M,false,White,4,3,3.3963,9.5000,M,null,null,null,null,Socks
710,"Mountain Bike Socks, L",SO-B909-L,false,White,4,3,3.3963,9.5000,L,null,null,null,null,Socks
711,"Sport-100 Helmet, Blue",HL-U509-B,false,Blue,4,3,13.0863,34.9900,null,null,null,null,null,Helmets
712,AWC Logo Cap,CA-1098,false,Multi,4,3,6.9223,8.9900,null,null,null,null,null,Caps
713,"Long-Sleeve Logo Jersey, S",LJ-0192-S,false,Multi,4,3,38.4923,49.9900,S,null,null,null,null,Jerseys
714,"Long-Sleeve Logo Jersey, M",LJ-0192-M,false,Multi,4,3,38.4923,49.9900,M,null,null,null,null,Jerseys


In [0]:
# Store sales order detail with proper data types
# PK: SalesOrderID + SalesOrderDetailID (composite)
# FK: SalesOrderID 
# FK: ProductID 
spark.sql("""
CREATE OR REPLACE TABLE enterprisedata_test.default.store_sales_order_detail AS
SELECT
  CAST(SalesOrderID AS INT) AS SalesOrderID,                    -- PK (composite),  --FK: SalesOrderHeader
  CAST(SalesOrderDetailID AS INT) AS SalesOrderDetailID,        -- PK (composite)
  CAST(OrderQty AS SHORT) AS OrderQty,
  CAST(ProductID AS INT) AS ProductID,                          -- FK : Products
  CAST(UnitPrice AS DECIMAL(10,4)) AS UnitPrice,
  CAST(UnitPriceDiscount AS DECIMAL(10,4)) AS UnitPriceDiscount
FROM raw_sales_order_detail
""")

display(spark.table("enterprisedata_test.default.store_sales_order_detail").limit(10))

SalesOrderID,SalesOrderDetailID,OrderQty,ProductID,UnitPrice,UnitPriceDiscount
43659,1,1,776,2024.9940,0.0000
43659,2,3,777,2024.9940,0.0000
43659,3,1,778,2024.9940,0.0000
43659,4,1,771,2039.9940,0.0000
43659,5,1,772,2039.9940,0.0000
43659,6,2,773,2039.9940,0.0000
43659,7,1,774,2039.9940,0.0000
43659,8,3,714,28.8404,0.0000
43659,9,1,716,28.8404,0.0000
43659,10,6,709,5.7000,0.0000


In [0]:
# Store sales order header with proper data types
# PK: SalesOrderID
# FK: CustomerID (external reference)
spark.sql("""
CREATE OR REPLACE TABLE enterprisedata_test.default.store_sales_order_header AS
SELECT
  CAST(SalesOrderID AS INT) AS SalesOrderID,   -- PK
  CAST(OrderDate AS DATE) AS OrderDate,
  CAST(ShipDate AS DATE) AS ShipDate,
  CAST(OnlineOrderFlag AS BOOLEAN) AS OnlineOrderFlag,
  CAST(AccountNumber AS STRING) AS AccountNumber,
  CAST(CustomerID AS INT) AS CustomerID, -- FK
  CAST(SalesPersonID AS INT) AS SalesPersonID,
  CAST(Freight AS DECIMAL(10,4)) AS Freight
FROM raw_sales_order_header
""")

display(spark.table("enterprisedata_test.default.store_sales_order_header").limit(10))

SalesOrderID,OrderDate,ShipDate,OnlineOrderFlag,AccountNumber,CustomerID,SalesPersonID,Freight
43828,2021-06-01,2021-07-05,true,10-4030-027605,27605,null,89.4568
43829,2021-06-01,2021-07-05,true,10-4030-027611,27611,null,89.4568
43830,2021-06-01,2021-07-05,true,10-4030-016347,16347,null,89.4568
43831,2021-06-01,2021-07-05,true,10-4030-011028,11028,null,84.3748
43832,2021-06-01,2021-07-05,true,10-4030-013584,13584,null,89.4568
43659,2021-05-31,2021-06-07,false,10-4020-000676,29825,279,616.0984
43660,2021-05-31,2021-06-07,false,10-4020-000117,29672,279,38.8276
43661,2021-05-31,2021-06-07,false,10-4020-000442,29734,282,985.5530
43662,2021-05-31,2021-06-07,false,10-4020-000227,29994,282,867.2389
43663,2021-05-31,2021-06-07,false,10-4020-000510,29565,276,12.5838


## Product Master Transformations
Replace NULL colors with 'N/A' and enhance missing ProductCategoryName based on subcategory logic.

In [0]:
# Publish product with NULL handling and category enhancement
spark.sql("""
CREATE OR REPLACE TABLE enterprisedata_test.default.publish_product AS
SELECT
  ProductID,
  ProductDesc,
  ProductNumber,
  MakeFlag,
  COALESCE(Color, 'N/A') AS Color,
  SafetyStockLevel,
  ReorderPoint,
  StandardCost,
  ListPrice,
  Size,
  SizeUnitMeasureCode,
  Weight,
  WeightUnitMeasureCode,
  CASE
    WHEN ProductCategoryName IS NOT NULL THEN ProductCategoryName
    WHEN ProductSubCategoryName IN ('Gloves', 'Shorts', 'Socks', 'Tights', 'Vests') THEN 'Clothing'
    WHEN ProductSubCategoryName IN ('Locks', 'Lights', 'Headsets', 'Helmets', 'Pedals', 'Pumps') THEN 'Accessories'
    WHEN ProductSubCategoryName LIKE '%Frames%' OR ProductSubCategoryName IN ('Wheels', 'Saddles') THEN 'Components'
    ELSE NULL  -- Explicitly NULL when no rule matches and original is NULL
  END AS ProductCategoryName,
  ProductSubCategoryName
FROM enterprisedata_test.default.store_products
""")

display(spark.table("enterprisedata_test.default.publish_product").limit(10))

ProductID,ProductDesc,ProductNumber,MakeFlag,Color,SafetyStockLevel,ReorderPoint,StandardCost,ListPrice,Size,SizeUnitMeasureCode,Weight,WeightUnitMeasureCode,ProductCategoryName,ProductSubCategoryName
680,"HL Road Frame - Black, 58",FR-R92B-58,true,Black,500,375,1059.3100,1431.5000,58,CM,2.24,LB,Components,Road Frames
706,"HL Road Frame - Red, 58",FR-R92R-58,true,Red,500,375,1059.3100,1431.5000,58,CM,2.24,LB,Components,Road Frames
707,"Sport-100 Helmet, Red",HL-U509-R,false,Red,4,3,13.0863,34.9900,null,null,null,null,Accessories,Helmets
708,"Sport-100 Helmet, Black",HL-U509,false,Black,4,3,13.0863,34.9900,null,null,null,null,Accessories,Helmets
709,"Mountain Bike Socks, M",SO-B909-M,false,White,4,3,3.3963,9.5000,M,null,null,null,Clothing,Socks
710,"Mountain Bike Socks, L",SO-B909-L,false,White,4,3,3.3963,9.5000,L,null,null,null,Clothing,Socks
711,"Sport-100 Helmet, Blue",HL-U509-B,false,Blue,4,3,13.0863,34.9900,null,null,null,null,Accessories,Helmets
712,AWC Logo Cap,CA-1098,false,Multi,4,3,6.9223,8.9900,null,null,null,null,null,Caps
713,"Long-Sleeve Logo Jersey, S",LJ-0192-S,false,Multi,4,3,38.4923,49.9900,S,null,null,null,null,Jerseys
714,"Long-Sleeve Logo Jersey, M",LJ-0192-M,false,Multi,4,3,38.4923,49.9900,M,null,null,null,null,Jerseys


## Sales Order Transformations
Join detail with header, calculate business-day lead time and extended price.

In [0]:
# Publish orders: join detail + header, compute business day lead time and extended price
# Formula: count inclusive weekdays between OrderDate and ShipDate, then subtract 1.
#   inclusive_weekdays = total_days + 1
#                       - saturdays_in_range
#                       - sundays_in_range
# Where:
#   saturdays = FLOOR((DATEDIFF + DAYOFWEEK(OrderDate)) / 7)
#   sundays   = FLOOR((DATEDIFF + DAYOFWEEK(OrderDate) - 1) / 7)
#   DAYOFWEEK: 1=Sun, 2=Mon, ..., 7=Sat
spark.sql("""
CREATE OR REPLACE TABLE enterprisedata_test.default.publish_orders AS
SELECT
  d.SalesOrderID,
  d.SalesOrderDetailID,
  d.OrderQty,
  d.ProductID,
  d.UnitPrice,
  d.UnitPriceDiscount,
  h.OrderDate,
  h.ShipDate,
  h.OnlineOrderFlag,
  h.AccountNumber,
  h.CustomerID,
  h.SalesPersonID,
  h.Freight AS TotalOrderFreight,
  -- Business day lead time: O(1) mathematical formula
  DATEDIFF(h.ShipDate, h.OrderDate)
    - CAST(FLOOR((DATEDIFF(h.ShipDate, h.OrderDate) + DAYOFWEEK(h.OrderDate)) / 7) AS INT)
    - CAST(FLOOR((DATEDIFF(h.ShipDate, h.OrderDate) + DAYOFWEEK(h.OrderDate) - 1) / 7) AS INT)
    AS LeadTimeInBusinessDays,
  ROUND(d.OrderQty * (d.UnitPrice - d.UnitPriceDiscount), 4) AS TotalLineExtendedPrice
FROM enterprisedata_test.default.store_sales_order_detail d
JOIN enterprisedata_test.default.store_sales_order_header h
  ON d.SalesOrderID = h.SalesOrderID
""")

display(spark.table("enterprisedata_test.default.publish_orders").limit(10))

SalesOrderID,SalesOrderDetailID,OrderQty,ProductID,UnitPrice,UnitPriceDiscount,OrderDate,ShipDate,OnlineOrderFlag,AccountNumber,CustomerID,SalesPersonID,TotalOrderFreight,LeadTimeInBusinessDays,TotalLineExtendedPrice
43659,1,1,776,2024.9940,0.0000,2021-05-31,2021-06-07,false,10-4020-000676,29825,279,616.0984,5,2024.9940
43659,2,3,777,2024.9940,0.0000,2021-05-31,2021-06-07,false,10-4020-000676,29825,279,616.0984,5,6074.9820
43659,3,1,778,2024.9940,0.0000,2021-05-31,2021-06-07,false,10-4020-000676,29825,279,616.0984,5,2024.9940
43659,4,1,771,2039.9940,0.0000,2021-05-31,2021-06-07,false,10-4020-000676,29825,279,616.0984,5,2039.9940
43659,5,1,772,2039.9940,0.0000,2021-05-31,2021-06-07,false,10-4020-000676,29825,279,616.0984,5,2039.9940
43659,6,2,773,2039.9940,0.0000,2021-05-31,2021-06-07,false,10-4020-000676,29825,279,616.0984,5,4079.9880
43659,7,1,774,2039.9940,0.0000,2021-05-31,2021-06-07,false,10-4020-000676,29825,279,616.0984,5,2039.9940
43659,8,3,714,28.8404,0.0000,2021-05-31,2021-06-07,false,10-4020-000676,29825,279,616.0984,5,86.5212
43659,9,1,716,28.8404,0.0000,2021-05-31,2021-06-07,false,10-4020-000676,29825,279,616.0984,5,28.8404
43659,10,6,709,5.7000,0.0000,2021-05-31,2021-06-07,false,10-4020-000676,29825,279,616.0984,5,34.2000


## Analysis
Answer key business questions using the transformed data.

In [0]:
%sql
-- Q1: Which color generated the highest revenue each year?
SELECT * FROM (
  SELECT
    YEAR(o.OrderDate) AS OrderYear,
    p.Color,
    ROUND(SUM(o.TotalLineExtendedPrice), 2) AS TotalRevenue,
    ROW_NUMBER() OVER (PARTITION BY YEAR(o.OrderDate) ORDER BY SUM(o.TotalLineExtendedPrice) DESC) AS rn
  FROM enterprisedata_test.default.publish_orders o
  JOIN enterprisedata_test.default.publish_product p ON o.ProductID = p.ProductID
  GROUP BY YEAR(o.OrderDate), p.Color
) ranked
WHERE rn = 1
ORDER BY OrderYear

OrderYear,Color,TotalRevenue,rn
2021,Red,6019614.02,1
2022,Black,14005242.98,1
2023,Black,15047694.37,1
2024,Yellow,6480746.07,1


In [0]:
%sql
-- Q2: What is the average LeadTimeInBusinessDays by ProductCategoryName?
SELECT
  p.ProductCategoryName,
  ROUND(AVG(o.LeadTimeInBusinessDays), 2) AS AvgLeadTimeInBusinessDays
FROM enterprisedata_test.default.publish_orders o
JOIN enterprisedata_test.default.publish_product p ON o.ProductID = p.ProductID
GROUP BY p.ProductCategoryName
ORDER BY AvgLeadTimeInBusinessDays DESC

ProductCategoryName,AvgLeadTimeInBusinessDays
Clothing,4.88
Accessories,4.88
null,4.87
Bikes,4.85
Components,4.85
